# Whisper fine-tuning (RU) — v2 with diagnostics

Same model, data and hyperparameters as v1, plus signals that show *why* a run goes wrong:

1. **Data check before training**: audio/text stats, leading-silence share, train/valid sentence overlap,
   kNN outliers in Whisper-encoder and text-embedding space, UMAP maps.
2. **Training diagnostics**: grad-norm spike/NaN warnings, per-layer update/weight ratio, per-layer activation std,
   baseline eval before training.
3. **Richer eval**: raw + normalized WER, CER, length ratio, empty/repetition rate, WER by duration bucket and by
   "sentence seen in train", printed sample transcripts, hallucination rate on synthetic non-speech audio.
4. **After training**: training curves, per-clip WER, UMAP of encoder embeddings before vs after (colored by WER),
   kNN-neighborhood overlap (forgetting check), worst clips and worst clusters.

Runs locally or on Kaggle (paths are detected). `SMOKE=1` env var runs a tiny version end-to-end.

In [ ]:
import importlib.util
import subprocess
import sys

for pkg, mod in [('umap-learn', 'umap'), ('jiwer', 'jiwer'), ('soxr', 'soxr'), ('matplotlib', 'matplotlib')]:
    if importlib.util.find_spec(mod) is None:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg], check=True)

In [ ]:
import glob
import math
import multiprocessing
import os
import re
from dataclasses import dataclass
from typing import Any, Dict, List

import jiwer
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import soundfile as sf
import soxr
import torch
import umap
from datasets import Dataset
from sklearn.cluster import KMeans
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.neighbors import NearestNeighbors
from transformers import (
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    TrainerCallback,
    WhisperForConditionalGeneration,
    WhisperProcessor,
)

multiprocessing.set_start_method('fork', force=True)  # DataLoader workers must inherit notebook-defined classes (py3.14 defaults to forkserver)
os.environ.setdefault('CUDA_VISIBLE_DEVICES', '0')  # single GPU: Kaggle T4x2 would otherwise trigger DataParallel

In [ ]:
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))
device = torch.device('cuda')

In [ ]:
ON_KAGGLE = os.path.exists('/kaggle/input')
if ON_KAGGLE:
    found = [p for pat in ('/kaggle/input/*/manifest.csv', '/kaggle/input/*/*/manifest.csv', '/kaggle/input/*/*/*/manifest.csv')
             for p in glob.glob(pat)]
    assert found, 'manifest.csv not found under /kaggle/input'
    CSV_PATH = found[0]
    AUDIO_DIR = os.path.join(os.path.dirname(CSV_PATH), '0')
    OUTPUT_DIR = '/kaggle/working/whisper-small-ru'
else:
    CSV_PATH = 'data/manifest.csv'
    AUDIO_DIR = 'data/0'
    OUTPUT_DIR = './whisper-small-ru-v2'
MODEL_NAME = 'openai/whisper-tiny'
N_SAMPLES = 10000
SAMPLING_RATE = 16000

SMOKE = os.environ.get('SMOKE') == '1'
MAX_STEPS = 10 if SMOKE else 2000
EVAL_STEPS = 5 if SMOKE else 500
N_VALID = 16 if SMOKE else 500
N_EMBED = 64 if SMOKE else 3000  # train clips used for the pre-training data map
DIAG_EVERY = 2 if SMOKE else 50  # steps between update-ratio / activation snapshots
NUM_WORKERS = 4

FIG_DIR = os.path.join(OUTPUT_DIR, 'figures')
os.makedirs(FIG_DIR, exist_ok=True)
print(CSV_PATH, AUDIO_DIR, OUTPUT_DIR, 'SMOKE' if SMOKE else '')

In [ ]:
# manifest.csv has unquoted commas in the text column, so split each line on the first comma only
with open(CSV_PATH, encoding='utf-8') as f:
    next(f)
    rows = [line.rstrip('\n').split(',', 1) for line in f]
df = pd.DataFrame(rows, columns=['filename', 'text'])
df = df[['filename', 'text']].dropna()
df['filename'] = df['filename'].apply(lambda x: os.path.join(AUDIO_DIR, x))
df = df[df['filename'].apply(os.path.exists)]
df = df.sample(n=min(N_SAMPLES, len(df)), random_state=42).reset_index(drop=True)
len(df)

In [ ]:
processor = WhisperProcessor.from_pretrained(MODEL_NAME, language='russian', task='transcribe')
model = WhisperForConditionalGeneration.from_pretrained(MODEL_NAME)
model.generation_config.language = 'russian'
model.generation_config.task = 'transcribe'
model.generation_config.forced_decoder_ids = None
model.config.use_cache = False

In [ ]:
def load_audio(path, sampling_rate=SAMPLING_RATE):
    audio, sr = sf.read(path, dtype='float32')
    if audio.ndim > 1:
        audio = audio.mean(axis=1)
    if sr != sampling_rate:
        audio = soxr.resample(audio, sr, sampling_rate, quality='LQ')
    return audio


@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any
    decoder_start_token_id: int
    sampling_rate: int = SAMPLING_RATE

    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:
        audios = [load_audio(f['filename'], self.sampling_rate) for f in features]
        batch = self.processor.feature_extractor(
            audios, sampling_rate=self.sampling_rate, return_tensors='pt'
        )

        labels_batch = self.processor.tokenizer(
            [f['text'] for f in features], padding=True, return_tensors='pt'
        )
        labels = labels_batch['input_ids'].masked_fill(labels_batch['attention_mask'].ne(1), -100)

        if (labels[:, 0] == self.decoder_start_token_id).all().cpu().item():
            labels = labels[:, 1:]

        batch['labels'] = labels
        return batch

In [ ]:
train_df, valid_df = train_test_split(df, test_size=0.1, random_state=42)
valid_df = valid_df.iloc[:N_VALID].reset_index(drop=True)
train_df = train_df.reset_index(drop=True)


def norm_text(s):
    s = s.lower().replace('ё', 'е')
    s = re.sub(r'[^\w\s]', ' ', s)
    return re.sub(r'\s+', ' ', s).strip()


# Common Voice has many speakers reading the same sentence: a random split puts the same text in train and valid
train_sentences = set(train_df['text'].map(norm_text))
valid_df['seen_in_train'] = valid_df['text'].map(norm_text).isin(train_sentences)
print(f"valid clips whose sentence also appears in train: {valid_df['seen_in_train'].mean():.1%}")
dup_counts = train_df['text'].map(norm_text).value_counts()
print(f'train sentences read more than once: {(dup_counts > 1).sum()} (max repeats: {dup_counts.max()})')

atc_dataset_train = Dataset.from_dict(train_df.to_dict(orient='list'))
atc_dataset_valid = Dataset.from_dict(valid_df[['filename', 'text']].to_dict(orient='list'))

## 1. Data check before training

Embed a sample of train clips + all valid clips with the **base** Whisper encoder (mean over the real-audio frames only,
not the 30 s padding). Outlier scores are computed in the original embedding space (kNN distance); UMAP is only for
looking. Things to look for: isolated islands (broken audio, wrong language), train and valid forming separate clouds
(domain shift), clusters of extreme speech rate (transcript doesn't match the audio).

In [ ]:
FRAME = int(0.02 * SAMPLING_RATE)


def audio_stats(a):
    n = len(a) // FRAME
    frame_rms = np.sqrt((a[: n * FRAME].reshape(n, FRAME) ** 2).mean(axis=1) + 1e-12) if n else np.zeros(1)
    voiced = np.flatnonzero(frame_rms > frame_rms.max() * 0.05)
    lead = voiced[0] * 0.02 if len(voiced) else len(a) / SAMPLING_RATE
    trail = (n - 1 - voiced[-1]) * 0.02 if len(voiced) else 0.0
    return {
        'duration': len(a) / SAMPLING_RATE,
        'rms_db': 20 * np.log10(np.sqrt((a ** 2).mean() + 1e-12)),
        'clip_frac': float((np.abs(a) > 0.99).mean()),
        'lead_silence': lead,
        'trail_silence': trail,
    }


class AudioOnly(torch.utils.data.Dataset):
    def __init__(self, paths):
        self.paths = list(paths)

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        return load_audio(self.paths[i])


@torch.no_grad()
def encoder_embeddings(model, paths, batch_size=32):
    """Mean-pooled last encoder layer over real-audio frames, plus per-clip audio stats.

    Whisper hidden states carry a strong per-position component, so a plain mean mostly encodes clip length.
    The average hidden state at each frame position (over these clips) is subtracted before pooling.
    """
    loader = torch.utils.data.DataLoader(AudioOnly(paths), batch_size=batch_size, num_workers=NUM_WORKERS, collate_fn=list)
    was_training = model.training
    model.eval().to(device)
    sums, counts, stats = [], [], []
    pos_sum, pos_count = None, None
    for audios in loader:
        stats += [audio_stats(a) for a in audios]
        feats = processor.feature_extractor(audios, sampling_rate=SAMPLING_RATE, return_tensors='pt').input_features
        with torch.autocast('cuda', dtype=torch.float16):
            hidden = model.model.encoder(feats.to(device)).last_hidden_state.float()
        n_frames = torch.tensor([min(hidden.shape[1], max(1, math.ceil(len(a) / SAMPLING_RATE / 0.02))) for a in audios], device=device)
        mask = (torch.arange(hidden.shape[1], device=device)[None] < n_frames[:, None]).float()
        masked = hidden * mask[..., None]
        pos_sum = masked.sum(0) if pos_sum is None else pos_sum + masked.sum(0)
        pos_count = mask.sum(0) if pos_count is None else pos_count + mask.sum(0)
        sums.append(masked.sum(1).cpu())
        counts.append(n_frames.cpu())
    model.train(was_training)
    pos_mean_cum = torch.cumsum(pos_sum / pos_count.clamp(min=1)[:, None], dim=0).cpu()
    counts = torch.cat(counts)
    emb = (torch.cat(sums) - pos_mean_cum[counts - 1]) / counts[:, None]
    return emb.numpy(), pd.DataFrame(stats)


def knn_outlier_score(x, k=10):
    x = x / np.linalg.norm(x, axis=1, keepdims=True)
    dist, idx = NearestNeighbors(n_neighbors=k + 1, metric='cosine').fit(x).kneighbors(x)
    return dist[:, 1:].mean(axis=1), dist[:, 1], idx[:, 1:]


def robust_z(v):
    v = np.asarray(v, dtype=float)
    mad = np.median(np.abs(v - np.median(v))) + 1e-9
    return (v - np.median(v)) / (1.4826 * mad)


def umap_2d(x, metric='cosine'):
    return umap.UMAP(n_neighbors=min(15, len(x) - 1), min_dist=0.1, metric=metric, random_state=42).fit_transform(x)


def scatter_grid(xy, color_cols, frame, title, path, ncols=2):
    nrows = math.ceil(len(color_cols) / ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(6 * ncols, 5 * nrows), squeeze=False)
    for ax, col in zip(axes.flat, color_cols):
        vals = frame[col]
        if not pd.api.types.is_numeric_dtype(vals) or pd.api.types.is_bool_dtype(vals):
            for v in sorted(vals.unique(), key=str):
                m = (vals == v).values
                ax.scatter(xy[m, 0], xy[m, 1], s=4, alpha=0.6, label=str(v))
            ax.legend(markerscale=3)
        else:
            lo, hi = np.nanpercentile(vals, [2, 98])
            sc = ax.scatter(xy[:, 0], xy[:, 1], c=vals, s=4, alpha=0.7, cmap='viridis', vmin=lo, vmax=hi)
            fig.colorbar(sc, ax=ax)
        ax.set_title(col)
        ax.set_xticks([])
        ax.set_yticks([])
    for ax in list(axes.flat)[len(color_cols):]:
        ax.axis('off')
    fig.suptitle(title)
    fig.tight_layout()
    fig.savefig(path, dpi=110)
    plt.show()

In [ ]:
check_df = pd.concat([
    train_df.sample(n=min(N_EMBED, len(train_df)), random_state=0).assign(split='train'),
    valid_df.drop(columns='seen_in_train').assign(split='valid'),
], ignore_index=True)

base_emb, base_stats = encoder_embeddings(model, check_df['filename'])
check_df = pd.concat([check_df, base_stats], axis=1)
check_df['n_words'] = check_df['text'].str.split().str.len()
check_df['chars_per_sec'] = check_df['text'].str.len() / check_df['duration'].clip(lower=0.1)
check_df['non_cyrillic'] = check_df['text'].str.count(r'[A-Za-z0-9]')
check_df['audio_outlier'], check_df['audio_nn_dist'], _ = knn_outlier_score(base_emb)
check_df['rate_z'] = robust_z(check_df['chars_per_sec'])

print(check_df[['duration', 'rms_db', 'lead_silence', 'trail_silence', 'chars_per_sec', 'n_words']].describe().round(2))
print(f"\nclips with <0.1s leading silence: {(check_df['lead_silence'] < 0.1).mean():.1%}  "
      "(models trained only on tightly-trimmed clips tend to hallucinate when real audio starts with silence)")
print(f"clips with clipping (>0.1% samples at full scale): {(check_df['clip_frac'] > 0.001).mean():.1%}")
print(f"texts with latin letters or digits: {(check_df['non_cyrillic'] > 0).mean():.1%}")
print(f"clips longer than 30s (truncated by Whisper): {(check_df['duration'] > 30).sum()}")

# Shift check: if valid looks like train, a valid clip's neighbours are valid at roughly the base rate
_, _, nn_idx = knn_outlier_score(base_emb)
is_valid = (check_df['split'] == 'valid').values
print(f"valid->valid neighbour share: {is_valid[nn_idx[is_valid]].mean():.1%} vs base rate {is_valid.mean():.1%}")

In [ ]:
base_xy = umap_2d(base_emb)
check_df['umap_x'], check_df['umap_y'] = base_xy[:, 0], base_xy[:, 1]
scatter_grid(base_xy, ['split', 'duration', 'chars_per_sec', 'audio_outlier', 'lead_silence', 'rms_db'], check_df,
             'Base Whisper encoder embeddings (UMAP)', os.path.join(FIG_DIR, 'umap_audio_before.png'), ncols=3)

In [ ]:
# Text map: char 2-4-gram TF-IDF -> SVD -> UMAP. Finds odd transcripts (other language, garbage, number formats)
tfidf = TfidfVectorizer(analyzer='char_wb', ngram_range=(2, 4), min_df=2, sublinear_tf=True).fit_transform(check_df['text'])
text_emb = TruncatedSVD(n_components=min(100, tfidf.shape[1] - 1), random_state=0).fit_transform(tfidf)
check_df['text_outlier'], _, _ = knn_outlier_score(text_emb)
text_xy = umap_2d(text_emb)
scatter_grid(text_xy, ['split', 'n_words', 'chars_per_sec', 'text_outlier'], check_df,
             'Transcript embeddings (TF-IDF, UMAP)', os.path.join(FIG_DIR, 'umap_text.png'))

In [ ]:
pd.set_option('display.max_colwidth', 120)
show = ['filename', 'text', 'duration', 'chars_per_sec', 'lead_silence']
check_df['short_name'] = check_df['filename'].map(os.path.basename)
show[0] = 'short_name'

print('Most isolated clips in audio-embedding space (listen to these):')
display(check_df.nlargest(10, 'audio_outlier')[show + ['audio_outlier']])
print('Closest audio pairs (possible duplicate recordings):')
display(check_df.nsmallest(5, 'audio_nn_dist')[show + ['audio_nn_dist']])
print('Most unusual transcripts:')
display(check_df.nlargest(10, 'text_outlier')[show + ['text_outlier']])
print('Extreme speech rate (|robust z| > 3, transcript may not match audio):')
display(check_df.loc[check_df['rate_z'].abs() > 3, show + ['rate_z']].sort_values('rate_z').head(15))

flags = check_df[(check_df['audio_outlier'] >= check_df['audio_outlier'].quantile(0.99))
                 | (check_df['text_outlier'] >= check_df['text_outlier'].quantile(0.99))
                 | (check_df['rate_z'].abs() > 3) | (check_df['clip_frac'] > 0.001)]
flags.drop(columns=['umap_x', 'umap_y']).to_csv(os.path.join(OUTPUT_DIR, 'data_flags.csv'), index=False)
print(f'{len(flags)} flagged clips saved to data_flags.csv')

## 2. Training with diagnostics

- `grad_norm` (logged by the Trainer, pre-clipping): warn on NaN/inf or a spike > 5x the recent median.
- **Update/weight ratio** per layer every `DIAG_EVERY` steps: healthy ≈ 1e-3 (log10 ≈ -3). Above -2 → LR too high
  for that layer; below -4 → layer barely learns. All-zero → fp16 overflow skipped the step.
- **Activation std** of every encoder/decoder layer output: exploding or collapsing values point to instability.
- Eval adds normalized WER/CER, length ratio, empty/repetition rate, WER buckets, sample transcripts and the
  hallucination rate on synthetic non-speech clips (silence, noise, hum, tones).

In [ ]:
def is_repetitive(s):
    words = s.split()
    if any(len(set(words[i:i + 4])) == 1 for i in range(len(words) - 3)):
        return True
    trigrams = [tuple(words[i:i + 3]) for i in range(len(words) - 2)]
    return bool(trigrams) and max(trigrams.count(t) for t in set(trigrams)) >= 3


def clip_wer(ref, hyp):
    return jiwer.wer(ref, hyp) if ref else np.nan


valid_duration = check_df.loc[check_df['split'] == 'valid', 'duration'].values
DURATION_BINS = [0, 3, 5, 8, 1000]
eval_samples_shown = list(range(min(4, N_VALID)))


def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id

    pred_str = processor.tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = processor.tokenizer.batch_decode(label_ids, skip_special_tokens=True)
    pred_norm = [norm_text(p) for p in pred_str]
    label_norm = [norm_text(r) for r in label_str]
    keep = [i for i, r in enumerate(label_norm) if r]

    per_clip = np.array([clip_wer(label_norm[i], pred_norm[i]) for i in range(len(label_norm))])
    metrics = {
        'wer': 100 * jiwer.wer(label_str, pred_str),
        'wer_norm': 100 * jiwer.wer([label_norm[i] for i in keep], [pred_norm[i] for i in keep]),
        'cer_norm': 100 * jiwer.cer([label_norm[i] for i in keep], [pred_norm[i] for i in keep]),
        'len_ratio': sum(len(p.split()) for p in pred_norm) / max(1, sum(len(r.split()) for r in label_norm)),
        'empty_rate': float(np.mean([not p for p in pred_norm])),
        'repeat_rate': float(np.mean([is_repetitive(p) for p in pred_norm])),
    }
    if len(valid_duration) == len(per_clip):
        buckets = pd.cut(valid_duration, DURATION_BINS)
        for b in buckets.categories:
            m = np.asarray(buckets == b)
            if m.any():
                metrics[f'wer_dur_{int(b.left)}-{int(min(b.right, 99))}s'] = 100 * np.nanmean(per_clip[m])
        seen = valid_df['seen_in_train'].values
        for flag, name in [(True, 'seen'), (False, 'unseen')]:
            if (seen == flag).any():
                metrics[f'wer_sentence_{name}'] = 100 * np.nanmean(per_clip[seen == flag])

    print('\n--- sample transcripts (fixed + worst) ---')
    worst = [int(i) for i in np.argsort(-np.nan_to_num(per_clip, nan=-1))[:3]]
    for i in dict.fromkeys(eval_samples_shown + worst):
        print(f'[{i}] wer={per_clip[i]:.2f}\n  REF: {label_str[i]}\n  HYP: {pred_str[i]}')
    return metrics

In [ ]:
LAYER_RE = re.compile(r'model\.(encoder|decoder)\.layers\.(\d+)\.')


def param_group(name):
    m = LAYER_RE.search(name)
    if m:
        return f'{m.group(1)[:3]}.L{int(m.group(2)):02d}'
    if 'encoder' in name:
        return 'enc.conv' if 'conv' in name else 'enc.other'
    return 'dec.embed' if 'embed' in name else 'dec.other'


class TrainingDiagnostics(TrainerCallback):
    def __init__(self, every=DIAG_EVERY, spike_factor=5.0):
        self.every, self.spike_factor = every, spike_factor
        self.snapshot, self.capture = None, False
        self.ratio_history, self.act_history, self.grad_norms = [], [], []
        self.act_std = {}

    def on_train_begin(self, args, state, control, model=None, **kwargs):
        for side, layers in [('enc', model.model.encoder.layers), ('dec', model.model.decoder.layers)]:
            for i, layer in enumerate(layers):
                layer.register_forward_hook(self._hook(f'{side}.L{i:02d}'))

    def _hook(self, name):
        def hook(module, inputs, output):
            if self.capture:
                out = output[0] if isinstance(output, tuple) else output
                self.act_std[name] = out.detach().float().std().item()
        return hook

    def on_step_begin(self, args, state, control, model=None, **kwargs):
        if (state.global_step + 1) % self.every == 0:
            self.snapshot = {n: p.detach().clone() for n, p in model.named_parameters() if p.requires_grad}
            self.capture, self.act_std = True, {}

    def on_step_end(self, args, state, control, model=None, **kwargs):
        if self.snapshot is None:
            return
        delta, weight = {}, {}
        for n, p in model.named_parameters():
            if n in self.snapshot:
                g = param_group(n)
                old = self.snapshot[n]
                delta[g] = delta.get(g, 0.0) + (p.detach() - old).abs().sum().item()
                weight[g] = weight.get(g, 0.0) + old.abs().sum().item()
        self.snapshot, self.capture = None, False
        step = state.global_step
        if all(v == 0 for v in delta.values()):
            print(f'[diag step {step}] optimizer step skipped (fp16 overflow) - no weights changed')
            return
        ratios = {g: math.log10(delta[g] / weight[g]) if delta[g] > 0 else float('-inf') for g in delta}
        self.ratio_history.append({'step': step, **ratios})
        self.act_history.append({'step': step, **self.act_std})
        too_high = [g for g, r in ratios.items() if r > -2]
        too_low = [g for g, r in ratios.items() if r < -4]
        enc = [r for g, r in ratios.items() if g.startswith('enc')]
        dec = [r for g, r in ratios.items() if g.startswith('dec')]
        acts = list(self.act_std.values()) or [float('nan')]
        print(f'[diag step {step}] log10(update/weight) enc {min(enc):.2f}..{max(enc):.2f} | dec {min(dec):.2f}..{max(dec):.2f}'
              f' | act std {min(acts):.2f}..{max(acts):.2f}'
              + (f' | HIGH (>-2): {too_high}' if too_high else '') + (f' | LOW (<-4): {too_low}' if too_low else ''))

    def on_log(self, args, state, control, logs=None, **kwargs):
        logs = logs or {}
        for key in ('loss', 'grad_norm'):
            if key in logs and not math.isfinite(logs[key]):
                print(f'WARNING step {state.global_step}: {key} is {logs[key]}')
        gn = logs.get('grad_norm')
        if gn is not None and math.isfinite(gn):
            recent = self.grad_norms[-20:]
            if len(recent) >= 5 and gn > self.spike_factor * np.median(recent):
                print(f'WARNING step {state.global_step}: grad_norm spike {gn:.2f} (recent median {np.median(recent):.2f})')
            self.grad_norms.append(gn)


def make_non_speech_clips(seconds=5.0, sr=SAMPLING_RATE, seed=0):
    rng = np.random.default_rng(seed)
    n = int(seconds * sr)
    t = np.arange(n) / sr
    white = rng.standard_normal(n)
    spectrum, freqs = np.fft.rfft(white), np.fft.rfftfreq(n)
    freqs[0] = freqs[1]
    pink = np.fft.irfft(spectrum / np.sqrt(freqs), n)
    brown = np.fft.irfft(spectrum / freqs, n)
    clicks = np.zeros(n)
    clicks[rng.integers(0, n, 20)] = 1.0
    tones = sum(np.sin(2 * np.pi * f * t) for f in (220, 277, 330)) * (np.sin(2 * np.pi * 0.5 * t) > 0)

    def at(x, db):
        return (x / (np.sqrt((x ** 2).mean()) + 1e-12) * 10 ** (db / 20)).astype(np.float32)

    return {
        'digital_silence': np.zeros(n, dtype=np.float32),
        'white_-60dB': at(white, -60), 'white_-30dB': at(white, -30),
        'pink_-35dB': at(pink, -35), 'brown_-30dB': at(brown, -30),
        'hum_50Hz': at(np.sin(2 * np.pi * 50 * t) + 0.3 * np.sin(2 * np.pi * 150 * t), -30),
        'clicks': at(clicks, -40), 'tones': at(tones, -25),
    }


class NonSpeechHallucinationCheck(TrainerCallback):
    """Any text produced on these clips is a hallucination (WER cannot see this)."""

    def __init__(self, processor):
        self.names, audios = zip(*make_non_speech_clips().items())
        self.features = processor.feature_extractor(list(audios), sampling_rate=SAMPLING_RATE, return_tensors='pt').input_features
        self.history = []

    @torch.no_grad()
    def run(self, model, step):
        was_training = model.training
        model.eval()
        with torch.autocast('cuda', dtype=torch.float16):
            ids = model.generate(self.features.to(model.device), max_new_tokens=64, use_cache=True)
        model.train(was_training)
        texts = processor.tokenizer.batch_decode(ids, skip_special_tokens=True)
        bad = {n: t.strip() for n, t in zip(self.names, texts) if norm_text(t)}
        rate = len(bad) / len(self.names)
        self.history.append({'step': step, 'nonspeech_halluc_rate': rate})
        print(f'[non-speech step {step}] hallucination rate {rate:.0%}' + ''.join(f'\n  {n}: {t[:100]!r}' for n, t in bad.items()))
        return rate

    def on_evaluate(self, args, state, control, model=None, **kwargs):
        self.run(model, state.global_step)

In [ ]:
data_collator = DataCollatorSpeechSeq2SeqWithPadding(
    processor=processor,
    decoder_start_token_id=model.config.decoder_start_token_id,
)

training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=1,
    gradient_checkpointing=True,
    fp16=True,
    learning_rate=1e-5,
    warmup_steps=min(200, MAX_STEPS // 5),
    max_steps=MAX_STEPS,
    eval_strategy='steps',
    eval_steps=EVAL_STEPS,
    save_steps=EVAL_STEPS,
    save_total_limit=2,
    logging_steps=min(25, EVAL_STEPS),
    logging_first_step=True,
    predict_with_generate=True,
    generation_max_length=225,
    load_best_model_at_end=True,
    metric_for_best_model='wer',
    greater_is_better=False,
    remove_unused_columns=False,
    dataloader_num_workers=NUM_WORKERS,
    dataloader_persistent_workers=True,
    report_to='none',
)

diagnostics = TrainingDiagnostics()
nonspeech = NonSpeechHallucinationCheck(processor)
trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=atc_dataset_train,
    eval_dataset=atc_dataset_valid,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    processing_class=processor.feature_extractor,
    callbacks=[diagnostics, nonspeech],
)

In [ ]:
# Baseline before any training: every later number is compared with this
baseline_metrics = trainer.evaluate()
print({k: round(v, 3) for k, v in baseline_metrics.items() if isinstance(v, float)})

In [ ]:
trainer.train()

In [ ]:
trainer.save_model(OUTPUT_DIR)
processor.save_pretrained(OUTPUT_DIR)

## 3. Training curves

In [ ]:
logs = pd.DataFrame(trainer.state.log_history)
train_logs = logs[logs['loss'].notna()] if 'loss' in logs else logs.iloc[0:0]
eval_logs = logs[logs['eval_wer'].notna()]
ratio_df = pd.DataFrame(diagnostics.ratio_history).set_index('step').sort_index(axis=1) if diagnostics.ratio_history else None
act_df = pd.DataFrame(diagnostics.act_history).set_index('step').sort_index(axis=1) if diagnostics.act_history else None
ns_df = pd.DataFrame(nonspeech.history)

fig, axes = plt.subplots(2, 3, figsize=(18, 9))
ax = axes[0, 0]
ax.plot(train_logs['step'], train_logs['loss'], label='train loss')
ax.plot(eval_logs['step'], eval_logs['eval_loss'], 'o-', label='eval loss')
ax.set_title('loss')
ax.legend()
ax = axes[0, 1]
ax.plot(train_logs['step'], train_logs['grad_norm'])
ax.set_yscale('log')
ax.set_title('grad norm (pre-clip)')
ax = axes[0, 2]
ax.plot(train_logs['step'], train_logs['learning_rate'])
ax.set_title('learning rate')
ax = axes[1, 0]
for col in ['eval_wer', 'eval_wer_norm', 'eval_cer_norm']:
    ax.plot(eval_logs['step'], eval_logs[col], 'o-', label=col)
ax.set_title('eval error rates (%)')
ax.legend()
ax = axes[1, 1]
for col in ['eval_len_ratio', 'eval_repeat_rate', 'eval_empty_rate']:
    ax.plot(eval_logs['step'], eval_logs[col], 'o-', label=col)
if len(ns_df):
    ax.plot(ns_df['step'], ns_df['nonspeech_halluc_rate'], 's--', label='non-speech halluc')
ax.set_title('output health')
ax.legend()
ax = axes[1, 2]
if ratio_df is not None:
    im = ax.imshow(ratio_df.T.values, aspect='auto', cmap='coolwarm', vmin=-5, vmax=-1,
                   extent=[ratio_df.index.min(), ratio_df.index.max(), len(ratio_df.columns) - 0.5, -0.5])
    ax.set_yticks(range(len(ratio_df.columns)))
    ax.set_yticklabels(ratio_df.columns, fontsize=6)
    fig.colorbar(im, ax=ax)
ax.set_title('log10(update / weight) per layer (target ≈ -3)')
fig.tight_layout()
fig.savefig(os.path.join(FIG_DIR, 'training_curves.png'), dpi=110)
plt.show()

if act_df is not None:
    act_df.plot(figsize=(12, 4), legend=False, title='activation std per layer output', logy=True)
    plt.savefig(os.path.join(FIG_DIR, 'activation_std.png'), dpi=110)
    plt.show()

wer_cols = [c for c in eval_logs.columns if c.startswith(('eval_wer_dur_', 'eval_wer_sentence_'))]
print('baseline vs end of training:')
print(pd.DataFrame({'baseline': pd.Series(baseline_metrics), 'last_eval': eval_logs.iloc[-1]}).loc[
    ['eval_wer', 'eval_wer_norm', 'eval_cer_norm', 'eval_len_ratio', 'eval_repeat_rate', 'eval_empty_rate'] + wer_cols].round(3))

## 4. After training: where does the model fail?

Per-clip WER of the best checkpoint, then the valid clips embedded by the **fine-tuned** encoder. High-WER regions on
the map are failure modes worth listening to. The kNN-overlap score says how much the encoder's notion of "similar
audio" changed: close to 1 = representation kept, low = large drift (possible forgetting of general knowledge).

In [ ]:
pred = trainer.predict(atc_dataset_valid)
pred_str = processor.tokenizer.batch_decode(pred.predictions, skip_special_tokens=True)
analysis = check_df[check_df['split'] == 'valid'].reset_index(drop=True).copy()
analysis['seen_in_train'] = valid_df['seen_in_train'].values
analysis['pred'] = pred_str
analysis['wer'] = [clip_wer(norm_text(r), norm_text(p)) for r, p in zip(analysis['text'], pred_str)]
analysis['repetitive'] = analysis['pred'].map(lambda p: is_repetitive(norm_text(p)))

after_emb, _ = encoder_embeddings(trainer.model, analysis['filename'])
before_emb = base_emb[(check_df['split'] == 'valid').values]


def knn_sets(x, k=10):
    x = x / np.linalg.norm(x, axis=1, keepdims=True)
    return NearestNeighbors(n_neighbors=k + 1, metric='cosine').fit(x).kneighbors(x)[1][:, 1:]


nb_before, nb_after = knn_sets(before_emb), knn_sets(after_emb)
overlap = np.mean([len(set(a) & set(b)) / len(set(a) | set(b)) for a, b in zip(nb_before, nb_after)])
print(f'kNN neighbourhood overlap before vs after fine-tuning (Jaccard, k=10): {overlap:.2f}')

xy_before, xy_after = umap_2d(before_emb), umap_2d(after_emb)
fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
lo, hi = 0, np.nanpercentile(analysis['wer'], 98)
for ax, xy, title in [(axes[0], xy_before, 'base encoder'), (axes[1], xy_after, 'fine-tuned encoder')]:
    sc = ax.scatter(xy[:, 0], xy[:, 1], c=analysis['wer'], s=8, cmap='magma_r', vmin=lo, vmax=hi)
    ax.set_title(f'{title} — color = per-clip WER (fine-tuned)')
    ax.set_xticks([])
    ax.set_yticks([])
fig.colorbar(sc, ax=axes)
fig.savefig(os.path.join(FIG_DIR, 'umap_valid_before_after.png'), dpi=110)
plt.show()

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, col in zip(axes, ['duration', 'chars_per_sec', 'lead_silence']):
    ax.scatter(analysis[col], analysis['wer'], s=6, alpha=0.5)
    ax.set_xlabel(col)
    ax.set_ylabel('WER')
fig.tight_layout()
fig.savefig(os.path.join(FIG_DIR, 'wer_vs_stats.png'), dpi=110)
plt.show()

In [ ]:
n_clusters = min(8, max(2, len(analysis) // 20))
analysis['cluster'] = KMeans(n_clusters=n_clusters, n_init=10, random_state=0).fit_predict(
    after_emb / np.linalg.norm(after_emb, axis=1, keepdims=True))
cluster_summary = analysis.groupby('cluster').agg(
    n=('wer', 'size'), mean_wer=('wer', 'mean'), duration=('duration', 'mean'),
    chars_per_sec=('chars_per_sec', 'mean'), lead_silence=('lead_silence', 'mean'), repetitive=('repetitive', 'mean'),
).sort_values('mean_wer', ascending=False)
print('Clusters of the fine-tuned embedding space, worst first:')
display(cluster_summary.round(3))

worst_cluster = cluster_summary.index[0]
print(f'Examples from worst cluster {worst_cluster}:')
display(analysis[analysis['cluster'] == worst_cluster].nlargest(8, 'wer')[['short_name', 'text', 'pred', 'wer', 'duration']])
print('Worst clips overall:')
display(analysis.nlargest(15, 'wer')[['short_name', 'text', 'pred', 'wer', 'duration', 'chars_per_sec', 'lead_silence', 'audio_outlier', 'seen_in_train']])

analysis.to_csv(os.path.join(OUTPUT_DIR, 'valid_analysis.csv'), index=False)
print('saved', os.path.join(OUTPUT_DIR, 'valid_analysis.csv'), 'and figures in', FIG_DIR)